# 1장. 랭체인을 활용한 LLM 기초 (LLM Fundamentals)

## 📌 1장 핵심 요약 (Chapter Summary)

이 장의 목표는 랭체인의 가장 기초가 되는 빌딩 블록들을 이해하고, 이를 조합하여 실제 작동하는 AI 체인을 만드는 것입니다.

1. **랭체인의 필요성:** 모델 독립성 확보, 재사용 가능한 패턴 활용, 복잡한 오케스트레이션 지원.
2. **모델 인터페이스:** 텍스트 기반의 **LLM**과 메시지 기반의 **채팅 모델(Chat Models)** 구분.
3. **프롬프트 관리:** 동적 입력을 처리하기 위한 **프롬프트 템플릿** 사용.
4. **데이터 구조화:** 모델의 응답을 원하는 형식(JSON, 리스트 등)으로 변환하는 **출력 파서**.
5. **표준 인터페이스:** 모든 컴포넌트가 공유하는 **Runnable** 인터페이스 (`invoke`, `batch`, `stream`).
6. **LCEL (LangChain Expression Language):** `|` 연산자를 이용한 선언적이고 효율적인 컴포넌트 결합 방식.

---

## LLM 애플리케이션 구축의 핵심 도전 과제

좋은 LLM 애플리케이션을 만드는 것은 단순히 모델에 질문을 던지는 것 이상을 의미합니다. 핵심 과제는 다음과 같습니다:
1. **프롬프트 구성:** 모델에 보낼 프롬프트를 효과적으로 구조화하는 것
2. **결과 처리:** 모델의 예측 결과를 처리하여 정확하고 유용한 출력을 반환하는 것

```
[사용자 입력] -> [프롬프트 구성(?)] -> [LLM] -> [출력 처리(?)] -> [최종 결과]
```

## 왜 랭체인(LangChain)인가?

각 모델 제공업체(OpenAI, Anthropic 등)의 SDK를 직접 사용할 수도 있지만, 랭체인을 사용하면 다음과 같은 이점이 있습니다:

- **일반적인 패턴의 사전 구현:** Chain-of-Thought, 도구 호출(Tool Calling) 등 자주 사용되는 패턴이 이미 구현되어 있습니다.
- **교체 가능한 빌딩 블록:** 모델, 출력 파서 등을 코드 수정 없이 쉽게 다른 것으로 교체할 수 있습니다.
- **업체 독립성:** OpenAI와 Anthropic은 메시지 형식이 미세하게 다르지만, 랭체인은 이를 추상화하여 동일한 코드로 여러 모델을 사용할 수 있게 해줍니다.
- **오케스트레이션 능력:** 관찰 가능성(Observability)을 위한 콜백 시스템, 중단 및 재시도 기능 등을 제공합니다.

In [ ]:
# ⏱ 패키지 설치 (1~2분 소요)
# langchain: LLM 애플리케이션 프레임워크
# langchain-ollama: Ollama 모델 연동 (ChatOllama, OllamaLLM)
# langchain-community: 서드파티 통합 도구
!pip install -q langchain langchain-ollama langchain-community

In [ ]:
# OpenAI API는 사용하지 않습니다 (Ollama 사용)
# from google.colab import userdata
# import os

# os.environ['OPENAI_API_KEY']=userdata.get('OPENAI_API_KEY')

In [ ]:
"""
⏱ Ollama 설치 및 모델 다운로드 (10~15분 소요, gpt-oss:20b ≈ 13GB)

[주의] Colab 환경 메모리 제약
- gpt-oss:20b는 약 13GB로 Colab 무료 티어 GPU(T4, VRAM 16GB)에는 메모리가 빠듯할 수 있음.
- 이 노트북은 먼저 Colab 환경에서 직접 실행해본 뒤, OOM(Out-Of-Memory) 오류가
  발생하면 gpt-oss:7b 같은 더 작은 변형으로 모델명을 바꾸는 편이 안전함.
- 모델명을 바꿀 경우 아래 `ollama pull ...`과 이후 모든 `ChatOllama(model=...)`,
  `OllamaLLM(model=...)` 호출의 모델명도 동일하게 교체해야 함.
"""
import subprocess
import time

!apt-get install -y zstd
!curl -fsSL https://ollama.com/install.sh | sh

subprocess.Popen(['ollama', 'serve'])
time.sleep(3)

!ollama pull gpt-oss:20b


---
## 1. LLM과 채팅 모델 (LLMs vs Chat Models)

랭체인은 두 가지 유형의 모델 인터페이스를 제공합니다:

- **LLM:** 문자열 프롬프트를 입력받아 문자열 응답을 반환하는 전통적인 인터페이스입니다. (예: `gpt-3.5-turbo-instruct`)
- **채팅 모델(Chat Models):** 메시지 목록을 입력받아 메시지 하나를 반환하는 인터페이스입니다. 최신 모델들은 대부분 이 방식을 선호합니다.

### 주요 설정 파라미터
- **temperature:** 출력의 무작위성을 조절합니다. 0에 가까울수록 결정론적(예측 가능)이고, 1에 가까울수록 창의적이고 예상치 못한 응답을 생성합니다.
- **max_tokens:** 생성될 응답의 최대 길이를 제한합니다.

### 코드 1-1 기본 LLM 호출
전통적인 문자열 기반의 인터페이스를 사용합니다.

In [ ]:
from langchain_ollama import OllamaLLM

model = OllamaLLM(model="gpt-oss:20b", temperature=0.1)

model.invoke("The sky is")

### 코드 1-2 채팅 모델 호출
대화형 인터페이스를 사용하며, 메시지 객체를 전달합니다.

In [ ]:
from langchain_ollama import ChatOllama
from langchain_core.messages import HumanMessage

model = ChatOllama(model="gpt-oss:20b")
prompt = [HumanMessage("What is the capital of France?")]

model.invoke(prompt)


---
## 2. 메시지 역할 (Message Roles)

채팅 모델은 메시지의 '역할'을 구분하여 처리합니다:

- **SystemMessage (System role):** AI에게 지침이나 페르소나를 부여합니다.
- **HumanMessage (User role):** 사용자의 질문이나 요청입니다.
- **AIMessage (Assistant role):** 모델의 응답입니다.
- **ChatMessage:** 임의의 역할을 설정할 수 있는 일반적인 메시지 객체입니다.

### 코드 1-3 시스템 메시지를 적용한 채팅 모델 호출
시스템 메시지를 통해 AI의 응답 스타일을 미리 설정할 수 있습니다.

In [ ]:
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_ollama import ChatOllama

model = ChatOllama(model="gpt-oss:20b")
system_msg = SystemMessage(
    '''You are a helpful assistant that responds to questions with three
        exclamation marks.'''
)
human_msg = HumanMessage('What is the capital of France?')

model.invoke([system_msg, human_msg])

---
## 3. 프롬프트 템플릿 (Prompt Templates)

프롬프트 템플릿은 정적인 텍스트와 동적인 입력값을 결합하여 모델에 보낼 최종 프롬프트를 만드는 '레시피'와 같습니다.

- **재사용성:** 동일한 구조의 프롬프트를 다른 데이터로 반복해서 만들 수 있습니다.
- **유연성:** 파이썬의 f-string 문법(`{variable}`)을 사용하여 런타임에 값을 채워넣습니다.

### 코드 1-4 프롬프트 템플릿 정의 및 호출

In [ ]:
from langchain_core.prompts import PromptTemplate

template = PromptTemplate.from_template("""Answer the question based on the
    context below. If the question cannot be answered using the information
    provided, answer with "I don't know".

Context: {context}

Question: {question}

Answer: """)

template.invoke({
    "context": """The most recent advancements in NLP are being driven by Large
        Language Models (LLMs). These models outperform their smaller
        counterparts and have become invaluable for developers who are creating
        applications with NLP capabilities. Developers can tap into these
        models through Hugging Face's `transformers` library, or by utilizing
        OpenAI and Cohere's offerings through the `openai` and `cohere`
        libraries, respectively.""",
    "question": "Which model providers offer LLMs?"
})

### 코드 1-5 동적 프롬프트와 모델 결합 호출

In [ ]:
from langchain_ollama import OllamaLLM
from langchain_core.prompts import PromptTemplate

template = PromptTemplate.from_template("""Answer the question based on the
    context below. If the question cannot be answered using the information
    provided, answer with "I don't know".

Context: {context}

Question: {question}

Answer: """)

model = OllamaLLM(model="gpt-oss:20b")

prompt = template.invoke({
    "context": """The most recent advancements in NLP are being driven by Large
        Language Models (LLMs). These models outperform their smaller
        counterparts and have become invaluable for developers who are creating
        applications with NLP capabilities. Developers can tap into these
        models through Hugging Face's `transformers` library, or by utilizing
        OpenAI and Cohere's offerings through the `openai` and `cohere`
        libraries, respectively.""",
    "question": "Which model providers offer LLMs?"
})

completion = model.invoke(prompt)

print(completion)

### 코드 1-6 역할 기반의 채팅 프롬프트 템플릿 (ChatPromptTemplate)
채팅 애플리케이션에서는 메시지 역할에 따라 동적 입력을 제공할 수 있는 `ChatPromptTemplate`을 사용합니다.

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
template = ChatPromptTemplate.from_messages([
    ('system', '''Answer the question based on the context below. If the
        question cannot be answered using the information provided, answer with
        "I don\'t know".'''),
    ('human', 'Context: {context}'),
    ('human', 'Question: {question}'),
])

template.invoke({
    "context": """The most recent advancements in NLP are being driven by Large
        Language Models (LLMs). These models outperform their smaller
        counterparts and have become invaluable for developers who are creating
        applications with NLP capabilities. Developers can tap into these
        models through Hugging Face's `transformers` library, or by utilizing
        OpenAI and Cohere's offerings through the `openai` and `cohere`
        libraries, respectively.""",
    "question": "Which model providers offer LLMs?"
})

### 코드 1-7 템플릿과 모델의 결합 호출
`ChatPromptTemplate`으로 만든 프롬프트를 채팅 모델에 직접 넘겨 한 번에 답을 받습니다. 템플릿과 모델 모두 재사용 가능한 컴포넌트입니다.

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_ollama import ChatOllama

template = ChatPromptTemplate.from_messages([
    ('system', '''Answer the question based on the context below. If the
        question cannot be answered using the information provided, answer
        with "I don\'t know".'''),
    ('human', 'Context: {context}'),
    ('human', 'Question: {question}'),
])

model = ChatOllama(model="gpt-oss:20b")

# template과 model 모두 여러 번 재사용 가능
prompt = template.invoke({
    "context": """The most recent advancements in NLP are being driven by
        Large Language Models (LLMs). These models outperform their smaller
        counterparts and have become invaluable for developers who are creating
        applications with NLP capabilities. Developers can tap into these
        models through Hugging Face's `transformers` library, or by utilizing
        OpenAI and Cohere's offerings through the `openai` and `cohere`
        libraries, respectively.""",
    "question": "Which model providers offer LLMs?"
})

model.invoke(prompt)


---
## 4. 구조화된 출력과 출력 파서 (Structured Outputs & Output Parsers)

단순 텍스트 응답뿐만 아니라 JSON, XML, CSV 등 기계가 읽을 수 있는 형식으로 결과를 얻어야 할 때가 있습니다.

- **JSON 출력:** 프론트엔드 코드나 데이터베이스에 저장하기에 적합합니다.
- **Pydantic 연동:** 파이썬에서는 `Pydantic` 라이브러리를 사용하여 출력 데이터의 스키마를 정의하고 검증할 수 있습니다.
- **출력 파서:** 모델의 텍스트 응답을 리스트나 딕셔너리 등 프로그래밍 객체로 변환해줍니다.

### 코드 1-8 JSON 형식으로 구조화된 출력 요청
`with_structured_output` 메소드를 사용하여 정의된 스키마에 맞는 결과를 얻을 수 있습니다.

In [ ]:
from langchain_ollama import ChatOllama
from pydantic import BaseModel

class AnswerWithJustification(BaseModel):
    '''사용자의 질문에 대한 답변과 그 이유를 포함하는 객체'''
    answer: str
    justification: str

llm = ChatOllama(model="gpt-oss:20b", temperature=0)
structured_llm = llm.with_structured_output(AnswerWithJustification)

result = structured_llm.invoke("What weighs more, a pound of bricks or a pound of feathers?")

print(result.model_dump_json(indent=2))


### 코드 1-9 CSV 출력 파서
텍스트를 리스트 형식으로 변환해주는 파서 예시입니다.

In [ ]:
from langchain_core.output_parsers import CommaSeparatedListOutputParser
parser = CommaSeparatedListOutputParser()
items = parser.invoke("apple, banana, cherry")
print(items)

---
## 5. Runnable 인터페이스와 LCEL (LangChain Expression Language)

랭체인의 모든 컴포넌트는 **Runnable 인터페이스**를 공유하므로 일관된 방식으로 사용할 수 있습니다:

- **invoke():** 단일 입력에 대해 응답을 생성합니다.
- **batch():** 여러 입력을 효율적으로 병렬 처리합니다.
- **stream():** 응답이 생성되는 대로 조각(Chunk)을 실시간으로 반환합니다.

### LCEL (선언적 구성)
`|` (파이프) 연산자를 사용하여 여러 컴포넌트를 연결하는 방식입니다. 
- **자동 병렬화:** 여러 작업을 알아서 최적화하여 실행합니다.
- **비동기 지원:** `ainvoke` 등을 통해 비동기 처리가 용이합니다.
- **스트리밍:** 전체 체인의 스트리밍을 쉽게 구현할 수 있습니다.

### 코드 1-10 랭체인의 공통 인터페이스 (invoke, batch, stream)

In [ ]:
from langchain_ollama import ChatOllama

model = ChatOllama(model="gpt-oss:20b")

print("--- invoke ---")
completion = model.invoke('Hi there!')
print(completion.content)

print("\n--- batch ---")
completions = model.batch(['Hi there!', 'Bye!'])
for res in completions:
    print(f"-> {res.content}")

print("\n--- stream ---")
for token in model.stream('Tell me a short joke.'):
    print(token.content, end="| ", flush=True)

### 코드 1-11 명령형 구성 (Imperative) 예시
`@chain` 데코레이터를 쓰면 일반 파이썬 함수에도 `invoke`, `stream`, `batch` 같은 Runnable 인터페이스가 자동으로 부여됩니다.

In [ ]:
from langchain_ollama import ChatOllama
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import chain

template = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant."),
    ("human", "{question}"),
])

model = ChatOllama(model="gpt-oss:20b")

# @chain 데코레이터: 함수에 Runnable 인터페이스를 부여
@chain
def chatbot(values):
    prompt = template.invoke(values)
    return model.invoke(prompt)

response = chatbot.invoke({"question": "Which model providers offer LLMs?"})
print(response.content)


### 코드 1-12 명령형 구성을 사용한 스트리밍
함수 내부에서 `model.stream(...)`으로 토큰 조각(chunk)을 `yield`하면, 호출부에서도 `chatbot.stream(...)`으로 점진적 응답을 받을 수 있습니다.

In [ ]:
@chain
def chatbot(values):
    prompt = template.invoke(values)
    for token in model.stream(prompt):
        yield token

for part in chatbot.stream({
    "question": "Which model providers offer LLMs?"
}):
    print(part.content, end="", flush=True)


### 코드 1-13 명령형 구성을 사용한 비동기 실행
`ainvoke`는 I/O 대기 시간이 많은 LLM 호출을 효율적으로 처리합니다. 함수를 `async def`로 정의하고 내부에서 `await`를 쓰면 됩니다.

In [ ]:
@chain
async def chatbot(values):
    prompt = await template.ainvoke(values)
    return await model.ainvoke(prompt)

response = await chatbot.ainvoke({"question": "Which model providers offer LLMs?"})
print(response.content)


### 코드 1-14 선언형 구성 (LCEL) 예시
`prompt | model`과 같이 선언적으로 체인을 구성합니다.

In [ ]:
from langchain_ollama import ChatOllama
from langchain_core.prompts import ChatPromptTemplate

template = ChatPromptTemplate.from_messages(
    [
        ("system", "You are a helpful assistant."),
        ("human", "{question}"),
    ]
)

model = ChatOllama(model="gpt-oss:20b")

# 파이프(|) 연산자를 사용한 체인 구성
chatbot = template | model

response = chatbot.invoke({"question": "Which model providers offer LLMs?"})
print(response.content)

### 코드 1-15 선언형 구성(LCEL)을 사용한 스트리밍
파이프(`|`) 연산자로 조립한 체인도 `stream`을 그대로 지원합니다.

In [ ]:
chatbot = template | model

for part in chatbot.stream({
    "question": "Which model providers offer LLMs?"
}):
    print(part.content, end="", flush=True)


### 코드 1-16 선언형 구성(LCEL)을 사용한 비동기 실행
LCEL 체인도 `ainvoke`를 통해 비동기 호출이 가능합니다.

In [ ]:
chatbot = template | model

response = await chatbot.ainvoke({
    "question": "Which model providers offer LLMs?"
})
print(response.content)


---
## 🏁 1장 전체 요약 및 결론 (Conclusion)

이 장에서는 랭체인을 사용하여 LLM 애플리케이션을 구축하는 데 필요한 핵심 구성 요소와 인터페이스를 학습했습니다.

### ✅ 핵심 학습 포인트
1. **체인(Chain)의 구조:** LLM 애플리케이션은 기본적으로 **프롬프트(입력 지침) + 모델(예측) + 출력 파서(변환)**의 체인으로 구성됩니다.
2. **일관된 인터페이스:** 모든 랭체인 컴포넌트는 `invoke`, `stream`, `batch` 메소드를 공유하여 사용법이 동일합니다.
3. **명령형 vs 선언형:** 익숙한 파이썬 함수 방식(명령형)과 LCEL 파이프 연산자 방식(선언형) 모두 가능하지만, **LCEL**이 복잡한 체인을 구성하고 최적화하는 데 더 유리합니다.
4. **확장성:** 랭체인의 추상화 덕분에 OpenAI에서 Ollama(로컬)로, 또는 그 반대로 모델을 교체하는 것이 매우 간단합니다.

### 🚀 다음 단계
지금까지는 모델이 가진 자체 지식만을 사용했습니다. **2장**에서는 외부 데이터(문서, 웹페이지 등)를 가져와 모델에게 **컨텍스트(Context)**로 제공함으로써, 모델이 모르는 정보에 대해서도 대답할 수 있게 만드는 방법을 학습합니다.